# 02 전처리

`01`의 6절 목록대로 정리해 `data/processed`에 세 표를 따로 저장한다. 커피-음료만 남긴다. 미리 합치지 않고, 연결은 SQL에서 한다.

In [1]:
from pathlib import Path

import pandas as pd

RAW = Path("../data/raw")
OUT = Path("../data/processed")
커피_코드 = "CS100010"
코드_열 = {"기준_년분기_코드": str, "상권_코드": str, "자치구_코드": str, "행정동_코드": str}


def 합치기(패턴, 이름_바꾸기=None):
    """연도별 파일을 읽어 열 이름을 맞춘 뒤 하나로 합친다."""
    조각 = []
    for 경로 in sorted(RAW.glob(패턴)):
        df = pd.read_csv(경로, encoding="cp949", dtype=코드_열)
        if 이름_바꾸기:
            df = df.rename(columns=이름_바꾸기)
        조각.append(df)
        print(경로.name, df.shape)
    return pd.concat(조각, ignore_index=True)

In [2]:
# 점포: 열 이름 통일 → 커피만
점포_이름 = {"점포_수": "일반_점포_수", "유사_업종_점포_수": "전체_점포_수"}
점포 = 합치기("*점포-상권*.csv", 점포_이름)
커피_점포 = 점포[점포["서비스_업종_코드"] == 커피_코드].copy()

# 매출: 시간대 건수 열 이름 수정 → 커피만
구간 = ["00~06", "06~11", "11~14", "14~17", "17~21", "21~24"]
매출_이름 = {f"시간대_건수~{g[-2:]}_매출_건수": f"시간대_{g}_매출_건수" for g in 구간}
매출 = 합치기("*추정매출*.csv", 매출_이름)
커피_매출 = 매출[매출["서비스_업종_코드"] == 커피_코드].copy()
print("커피 매출", len(커피_매출), "/ 커피 점포", len(커피_점포))

서울시_상권분석서비스(점포-상권)_2021년.csv (303880, 14)


서울시_상권분석서비스(점포-상권)_2022년.csv (305587, 14)


서울시_상권분석서비스(점포-상권)_2023년.csv (307741, 14)


서울시_상권분석서비스(점포-상권)_2024년.csv (306889, 14)


서울시_상권분석서비스(점포-상권)_2025~2026년.csv (456659, 14)


서울시_상권분석서비스(추정매출-상권)_2021년.csv (89150, 55)


서울시_상권분석서비스(추정매출-상권)_2022년.csv (88834, 55)


서울시_상권분석서비스(추정매출-상권)_2023년.csv (88246, 55)


서울시_상권분석서비스(추정매출-상권)_2024년.csv (87179, 55)


서울시_상권분석서비스(추정매출-상권)_2025~2026년.csv (128053, 55)
커피 매출 23700 / 커피 점포 33263


In [3]:
# 영역: 깨진 가운뎃점 복구, 상권 이름은 영역 표 기준으로 통일
영역 = pd.read_csv(RAW / "서울시_상권분석서비스(영역-상권).csv", encoding="cp949", dtype=코드_열)
for 열 in ["상권_코드_명", "행정동_코드_명"]:
    영역[열] = 영역[열].str.replace("?", "·", regex=False)

기준_이름 = 영역.set_index("상권_코드")["상권_코드_명"]
for df in (커피_매출, 커피_점포):
    df["상권_코드_명"] = df["상권_코드"].map(기준_이름)

In [4]:
# 정렬, 실수형 금액 → 정수
키 = ["기준_년분기_코드", "상권_코드"]
커피_매출 = 커피_매출.sort_values(키).reset_index(drop=True)
커피_점포 = 커피_점포.sort_values(키).reset_index(drop=True)
실수_열 = 커피_매출.select_dtypes("float").columns
assert (커피_매출[실수_열] % 1 == 0).all().all()  # 소수 부분이 있으면 멈춘다
커피_매출[실수_열] = 커피_매출[실수_열].astype("int64")

In [5]:
# 검증: 처리 결과가 01에서 본 문제를 모두 해결했는가
점검 = {
    "매출 키 중복": 커피_매출.duplicated(키).sum(),
    "점포 키 중복": 커피_점포.duplicated(키).sum(),
    "상권 이름 빈 값": 커피_매출["상권_코드_명"].isna().sum() + 커피_점포["상권_코드_명"].isna().sum(),
    "이름에 '?' 남음": 영역["상권_코드_명"].str.contains("?", regex=False).sum() + 영역["행정동_코드_명"].str.contains("?", regex=False).sum(),
    "이름이 2개인 상권": (커피_매출.groupby("상권_코드")["상권_코드_명"].nunique() > 1).sum(),
    "깨진 시간대 열": sum(열.startswith("시간대_건수") for 열 in 커피_매출.columns),
    "실수형 열": len(커피_매출.select_dtypes("float").columns),
}
assert all(v == 0 for v in 점검.values()), 점검
점검

{'매출 키 중복': np.int64(0),
 '점포 키 중복': np.int64(0),
 '상권 이름 빈 값': np.int64(0),
 "이름에 '?' 남음": np.int64(0),
 '이름이 2개인 상권': np.int64(0),
 '깨진 시간대 열': 0,
 '실수형 열': 0}

In [6]:
OUT.mkdir(parents=True, exist_ok=True)
for 이름, df in {"커피_매출": 커피_매출, "커피_점포": 커피_점포, "영역": 영역}.items():
    df.to_csv(OUT / f"{이름}.csv", index=False, encoding="utf-8")
    다시 = pd.read_csv(OUT / f"{이름}.csv", dtype=코드_열)
    print(이름, df.shape, "| 다시 읽은 행 수 일치:", len(다시) == len(df))

커피_매출 (23700, 55) | 다시 읽은 행 수 일치: True
커피_점포 (33263, 14) | 다시 읽은 행 수 일치: True
영역 (1650, 11) | 다시 읽은 행 수 일치: True
